# LABORATORIO N.° 03 — La métrica que importa

**Curso:** Analítica Empresarial Integrada  
**Semana 3:** KPI accionables, North Star Metric y árbol de métricas  
**Docente:** Pilar Rocío Sayán Mejía  
**Periodo:** 2026-II

**Fuente real:** UCI Machine Learning Repository — Online Retail (ID 352).

> Objetivo del notebook: conectar **objetivo → North Star → drivers → guardrails → KPI → tablero → interpretación → decisión** usando transacciones reales.
---

**Sección:** 6C28 &nbsp;&nbsp;&nbsp; **Fecha:** 02/09/2026

**Apellidos y nombres del estudiante:** Rodrigo Alejandro Gogin Cisterna



---


## Agenda de laboratorio — 7:00 p. m. a 10:10 p. m.

**Duración total:** 190 minutos · **Receso:** 15 minutos · **Trabajo efectivo:** 175 minutos.

| Horario | Tiempo | Desarrollo |
|---|---:|---|
| 7:00–7:10 | 10 min | Apertura del caso y presentación del problema de medición. |
| 7:10–7:35 | 25 min | Actividad 1. Revisión de conceptos: del dato a la decisión. |
| 7:35–8:00 | 25 min | Actividad 2. Descarga desde UCI, auditoría y limpieza documentada. |
| 8:00–8:30 | 30 min | Actividad 2. Periodo comparable, recurrencia y North Star. Reto 1. |
| 8:30–8:45 | 15 min | **RECESO** |
| 8:45–9:15 | 30 min | Actividad 2. Árbol de métricas, guardrails, Polars y DuckDB. |
| 9:15–9:35 | 20 min | Actividad 2. Tablero de decisión en Plotly. Reto 2. |
| 9:35–10:00 | 25 min | **Reto de aplicación y retroalimentación.** Ejercicios 1 a 5. |
| 10:00–10:10 | 10 min | Diccionario de KPI, informe ejecutivo y ticket de salida. |

> **Regla de trabajo:** no avance de bloque sin registrar la interpretación solicitada. El objetivo no es ejecutar celdas, sino convertir datos en evidencia para una decisión.


## Actividad 1 — Revisión de conceptos: del dato a la decisión (25 minutos)

**Propósito.** Establecer con precisión el vocabulario de medición antes de programar. La confusión entre dato, métrica, indicador y KPI constituye la causa más frecuente de tableros que no sustentan ninguna decisión.

**Instrucciones.** Complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La columna de la derecha contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

**Evidencia esperada.** Tabla completa con las definiciones registradas.

| Concepto | Definición elaborada por el estudiante | Pregunta de apoyo |
|---|---|---|
| Dato | Un valor puntual, sin procesar, registrado en una transacción individual (por ejemplo, `Quantity = 6` en la factura 536365). Por sí solo no dice nada sobre el desempeño del negocio. | ¿En qué se diferencia un dato de una métrica? Proponga un ejemplo de la base utilizada. |
| Métrica | El resultado de agregar o calcular matemáticamente uno o varios datos (por ejemplo, la suma de `importe_linea` de todas las facturas de un mes). Describe una magnitud, pero todavía no dice si esa magnitud es buena o mala. | ¿Toda métrica calculada correctamente resulta útil para decidir? Justifique. |
| Indicador | Una métrica a la que se le agrega contexto de comparación (una meta, un periodo anterior o un umbral), de modo que permite emitir un juicio sobre el desempeño. | ¿Qué debe añadirse a una métrica para que constituya un indicador? |
| KPI | Un indicador seleccionado, entre muchos posibles, porque está directamente ligado a un objetivo estratégico y orienta una decisión concreta y periódica del negocio. | ¿Por qué una organización no puede sostener veinte KPI simultáneos? |
| Meta | El valor que el equipo se propone alcanzar en un periodo futuro, distinto del valor observado en los datos históricos; es una aspiración de gestión, no un hecho ya ocurrido. | ¿Qué distingue un valor observado en la base de una meta propuesta para el ejercicio? |
| North Star | La única métrica que mejor resume el valor real entregado al cliente y que, si mejora de forma sostenida, se traduce en éxito del negocio a largo plazo. | ¿Qué condición debe cumplir una North Star para no convertirse en métrica de vanidad? |
| Driver | Una variable intermedia que explica el movimiento de la North Star y sobre la cual el equipo puede actuar de manera directa (por ejemplo, la cantidad de clientes recurrentes). | ¿Cómo se comprueba que una variable es efectivamente driver de la North Star? |
| Guardrail | Una métrica de control que se vigila en paralelo a la North Star para asegurar que, al optimizarla, no se esté deteriorando otro aspecto crítico del negocio (calidad, riesgo, concentración). | ¿Qué ocurre si una organización optimiza su North Star sin vigilar los guardrails? |

**Criterio de cierre.** No se avanza al desarrollo práctico mientras existan conceptos sin definición registrada.


## Actividad 2 — Desarrollo práctico y ejecución

### Presentación del caso

Una empresa minorista en línea del Reino Unido cuenta con un registro histórico de transacciones que incluye facturas, productos, cantidades, fechas, precios unitarios, clientes y países. La dirección necesita transformar estos registros en un sistema de métricas que permita distinguir crecimiento útil de crecimiento aparente. Para ello, no basta con observar ventas acumuladas o cantidad de clientes: es necesario identificar qué indicadores representan valor recurrente para el cliente y cuáles pueden orientar una decisión empresarial.

Durante el laboratorio, el equipo trabajará con el conjunto Online Retail del UCI Machine Learning Repository. A partir de los datos reales, deberá auditar y preparar las transacciones, identificar clientes recurrentes, proponer y justificar una North Star, descomponerla en drivers y guardrails, construir un diccionario de KPI y elaborar un tablero de decisión en Plotly. El análisis deberá terminar con hallazgos cuantitativos y una acción empresarial concreta, diferenciando en todo momento los valores observados en la base de las metas o umbrales académicos propuestos para el ejercicio.


## Paso 1 — Preparación reproducible del entorno


In [1]:
%pip install -q ucimlrepo==0.0.7 polars==1.17.1 duckdb==1.1.3

import polars as pl
import duckdb
import pandas as pd            # solo para recibir la descarga de UCI y alimentar Plotly Express
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from ucimlrepo import fetch_ucirepo

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)

# Formato REAL de las fechas de esta fuente: 12/1/2010 8:26 -> mes/dia/anio hora:min.
# Se declara de forma explicita en lugar de dejar que la libreria lo adivine.
FORMATO_FECHA = "%m/%d/%Y %H:%M"

print("polars:", pl.__version__, "| duckdb:", duckdb.__version__)
print("Entorno listo.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 52.3 MB/s eta 0:00:00
polars: 1.17.1 | duckdb: 1.1.3
Entorno listo.


## Paso 2 — Descarga de datos reales desde UCI

El conjunto **Online Retail** contiene transacciones de una empresa minorista en línea registrada en el Reino Unido entre diciembre de 2010 y diciembre de 2011. Los códigos de factura que empiezan con `C` representan cancelaciones. No se generan registros artificiales.


In [2]:
online_retail = fetch_ucirepo(id=352)
df = pl.from_pandas(online_retail.data.original)   # la descarga llega en pandas; se pasa a Polars

print("Dataset:", online_retail.metadata.get("name"))
print("Filas y columnas:", df.shape)
print("Columnas:", df.columns)
display(df.head())


Dataset: Online Retail
Filas y columnas: (541909, 8)
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,str,f64,f64,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,"""12/1/2010 8:26""",2.55,17850.0,"""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,"""12/1/2010 8:26""",2.75,17850.0,"""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""


### Control de trazabilidad
Registre: número de filas, columnas, rango de fechas y países presentes.

**Respuesta:**

El dataset descargado desde UCI (ID 352) contiene **541,909 filas** y **8 columnas** (`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`). El rango de fechas va del **2010-12-01 08:26** al **2011-12-09 12:50**. Según la documentación oficial del dataset, las transacciones proceden de un comercio con base en el Reino Unido y abarcan **38 países** distintos, aunque el Reino Unido concentra la gran mayoría de las facturas.


## Paso 3 — Auditoría inicial y limpieza documentada


In [3]:
df = df.with_columns([
    pl.col("InvoiceNo").cast(pl.Utf8),
    pl.col("CustomerID").cast(pl.Utf8),
    pl.col("Quantity").cast(pl.Float64, strict=False),
    pl.col("UnitPrice").cast(pl.Float64, strict=False),
    pl.col("InvoiceDate").cast(pl.Utf8)
      .str.to_datetime(format=FORMATO_FECHA, strict=False).alias("InvoiceDate"),
])
df = df.with_columns([
    pl.col("InvoiceNo").str.to_uppercase().str.starts_with("C").alias("es_cancelacion"),
    (pl.col("Quantity") * pl.col("UnitPrice")).alias("importe_linea"),
])

# Control de parseo: si el formato declarado no fuese el correcto, aqui apareceria
# un conteo de fechas nulas. pandas habria "adivinado" un formato sin avisar;
# Polars obliga a declararlo y deja el error a la vista.
nulas = df["InvoiceDate"].null_count()
print("Fechas que no pudieron parsearse:", nulas)
if nulas:
    raise ValueError("El formato declarado en FORMATO_FECHA no corresponde a la fuente.")

# Control de integridad de la fuente. Si UCI republica el archivo, el laboratorio
# se detiene aqui en vez de producir numeros que no coinciden con el solucionario.
FILAS_ESPERADAS = 541_909
FACTURAS_ESPERADAS = 25_900

if df.height != FILAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FILAS_ESPERADAS:,} filas y llegaron {df.height:,}. "
        "Avise al docente antes de continuar."
    )
if df["InvoiceNo"].n_unique() != FACTURAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FACTURAS_ESPERADAS:,} facturas distintas "
        f"y llegaron {df['InvoiceNo'].n_unique():,}."
    )
print("Integridad verificada:", f"{df.height:,}", "filas y",
      f"{df['InvoiceNo'].n_unique():,}", "facturas, como se esperaba.")

control = pl.DataFrame({
    "indicador": ["filas", "facturas", "clientes", "cancelaciones",
                  "cantidades_no_positivas", "precios_no_positivos"],
    "valor": [df.height,
              df["InvoiceNo"].n_unique(),
              df["CustomerID"].drop_nulls().n_unique(),
              int(df["es_cancelacion"].sum()),
              int((df["Quantity"] <= 0).sum()),
              int((df["UnitPrice"] <= 0).sum())],
})
display(control)
print("Rango:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())


Fechas que no pudieron parsearse: 0
Integridad verificada: 541,909 filas y 25,900 facturas, como se esperaba.


indicador,valor
str,i64
"""filas""",541909
"""facturas""",25900
"""clientes""",4372
"""cancelaciones""",9288
"""cantidades_no_positivas""",10624
"""precios_no_positivos""",2517


Rango: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [4]:
compras = df.filter(
    (~pl.col("es_cancelacion"))
    & (pl.col("Quantity") > 0)
    & (pl.col("UnitPrice") > 0)
    & pl.col("InvoiceDate").is_not_null()
    & pl.col("CustomerID").is_not_null()
)

print("Filas originales:", df.height)
print("Filas de compra validas:", compras.height)
print("Proporcion conservada: {:.1%}".format(compras.height / df.height))


Filas originales: 541909
Filas de compra validas: 397884
Proporcion conservada: 73.4%


### Reto de calidad
1. ¿Qué sesgo aparece si contamos cancelaciones como ventas?  
2. ¿Por qué no debemos borrar esos registros de la fuente original?

**Respuesta:**

1. Si se cuentan las cancelaciones (facturas que empiezan con `C`, con `Quantity` negativa) como ventas, los ingresos y las unidades vendidas quedan **sobreestimados**: se estaría reconociendo como ingreso algo que el cliente terminó devolviendo, lo que infla artificialmente cualquier North Star basada en ingresos o unidades.
2. No deben borrarse de la fuente original porque son evidencia real del comportamiento del cliente (tasa de devoluciones, calidad del catálogo, fricción del proceso de compra) y sirven como insumo directo para el guardrail de tasa de cancelación; borrarlas eliminaría la posibilidad de auditar y de calcular ese guardrail más adelante.


## Paso 4 — Definición del periodo comparable

La fuente empieza el 1/12/2010 y termina el 9/12/2011. Para no comparar meses incompletos con meses completos, el laboratorio informa sobre **enero–noviembre de 2011**.

**Cuidado con el sesgo de ventana.** La tabla de facturas se construye sobre *todo* el historial disponible, y el recorte a la ventana se aplica **después** de determinar la recurrencia. Si se recorta antes, un cliente que compró en diciembre de 2010 y volvió en enero de 2011 aparece como comprador nuevo, y la North Star crece de forma artificial en los primeros meses: en esta base, enero pasa de 570 a 246 compras recurrentes, un 57 % menos, solo por haber filtrado en el orden equivocado.


In [5]:
compras = compras.with_columns(pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes"))
INICIO, FIN = pl.datetime(2011, 1, 1), pl.datetime(2011, 12, 1)

# La tabla se construye sobre TODO el historial: el recorte a la ventana
# comparable se hace mas adelante, una vez determinada la recurrencia.
facturas = (compras.group_by(["InvoiceNo", "CustomerID", "mes"])
            .agg([pl.col("InvoiceDate").min().alias("fecha_factura"),
                  pl.col("importe_linea").sum().alias("importe_factura"),
                  pl.col("Quantity").sum().alias("unidades"),
                  pl.col("StockCode").count().alias("lineas")]))

en_ventana = facturas.filter((pl.col("fecha_factura") >= INICIO) & (pl.col("fecha_factura") < FIN))
print("Facturas en el historial completo :", facturas["InvoiceNo"].n_unique())
print("Facturas en la ventana comparable :", en_ventana["InvoiceNo"].n_unique())
print("Clientes en la ventana            :", en_ventana["CustomerID"].n_unique())
print("Ingresos en la ventana (GBP)      :", round(en_ventana["importe_factura"].sum(), 2))
display(facturas.head())


Facturas en el historial completo : 18532
Facturas en la ventana comparable : 16354
Clientes en la ventana            : 4173
Ingresos en la ventana (GBP)      : 7820501.22


InvoiceNo,CustomerID,mes,fecha_factura,importe_factura,unidades,lineas
str,str,str,datetime[μs],f64,f64,u32
"""568708""","""12393.0""","""2011-09""",2011-09-28 15:41:00,521.5,294.0,25
"""573930""","""17811.0""","""2011-11""",2011-11-02 10:15:00,120.66,125.0,9
"""550659""","""17730.0""","""2011-04""",2011-04-20 09:02:00,347.17,296.0,19
"""552249""","""12748.0""","""2011-05""",2011-05-06 19:53:00,35.97,18.0,11
"""575069""","""17637.0""","""2011-11""",2011-11-08 12:51:00,208.11,65.0,11


## Paso 5 — Recurrencia y North Star

**Regla operativa del laboratorio:** un cliente se considera recurrente desde su segunda factura válida, contada sobre **todo el historial disponible**. Su primera compra no se reclasifica retrospectivamente.

Enero de 2011 conserva un sesgo residual, porque solo dispone de un mes previo de historial. Por eso se marca como **mes de calentamiento** y se excluye de las comparaciones de variación mensual.


In [6]:
facturas = facturas.sort(["CustomerID", "fecha_factura", "InvoiceNo"])
facturas = facturas.with_columns(
    (pl.int_range(pl.len()).over("CustomerID") + 1).alias("n_compra_cliente"))

# Una factura es recurrente a partir de la SEGUNDA compra del cliente, en orden
# cronologico. La marca se calcula sobre el historial completo para no clasificar
# retroactivamente como recurrente la primera compra de un cliente que volvio despues.
facturas = facturas.with_columns(
    (pl.col("n_compra_cliente") >= 2).alias("es_recurrente"))

# Recien ahora se recorta a la ventana comparable: la recurrencia ya quedo
# determinada usando todo el historial, sin sesgo de ventana.
facturas = facturas.filter((pl.col("fecha_factura") >= INICIO) & (pl.col("fecha_factura") < FIN))

# Enero solo tiene un mes previo de historial: no es comparable en variacion.
MES_CALENTAMIENTO = "2011-01"

mensual = (facturas.group_by("mes")
           .agg([pl.col("InvoiceNo").n_unique().alias("facturas_validas"),
                 pl.col("CustomerID").n_unique().alias("clientes_activos"),
                 pl.col("importe_factura").sum().alias("ingresos")]))

rec = (facturas.filter("es_recurrente").group_by("mes")
       .agg([pl.col("InvoiceNo").n_unique().alias("compras_recurrentes"),
             pl.col("CustomerID").n_unique().alias("clientes_recurrentes"),
             pl.col("importe_factura").sum().alias("ingresos_recurrentes")]))

kpi = mensual.join(rec, on="mes", how="left").fill_null(0).sort("mes")
kpi = kpi.with_columns([
    (pl.col("compras_recurrentes") / pl.col("clientes_recurrentes")).alias("frecuencia_recurrente"),
    (100 * pl.col("ingresos_recurrentes") / pl.col("ingresos")).alias("participacion_ingreso_recurrente_pct"),
])
display(kpi)


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct
str,u32,u32,f64,u32,u32,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535
"""2011-06""",1393,991,661213.69,1151,773,570140.0,1.489004,86.226285
"""2011-07""",1331,949,600091.011,1143,781,532074.68,1.463508,88.665664
"""2011-08""",1280,935,645343.9,1111,778,568263.68,1.428021,88.055947
"""2011-09""",1755,1266,952838.382,1456,996,806684.471,1.461847,84.661207


### North Star propuesta
**Compras válidas de clientes recurrentes por mes.**

Justifique con los cuatro criterios: valor para el cliente, vínculo con valor empresarial, capacidad de influencia del equipo y descomposición en drivers.

**Respuesta:**

- **Valor para el cliente:** un cliente que vuelve a comprar es un cliente que encontró algo útil en su primera experiencia; contar sus compras válidas mide, indirectamente, la satisfacción repetida.
- **Vínculo con valor empresarial:** en la ventana analizada, la participación del ingreso proveniente de compras recurrentes pasó de 52% a 90% del ingreso mensual, es decir, la salud del negocio depende cada vez más de este grupo.
- **Capacidad de influencia del equipo:** marketing y CRM pueden actuar directamente sobre la recurrencia (fidelización, remarketing, ofertas), a diferencia de métricas como "ingresos totales", que dependen también de factores externos (estacionalidad, tráfico nuevo).
- **Descomposición en drivers:** la métrica se descompone de forma exacta como `clientes_recurrentes × frecuencia_recurrente`, lo que permite saber si hay que atraer más clientes que vuelvan o lograr que cada uno compre con mayor frecuencia.


### Reto 1 — ¿Vanidad o acción?
Clasifique: productos totales, clientes activos mensuales, compras recurrentes, ingresos acumulados, tasa de cancelación y países con ventas. Para cada una indique qué decisión permite tomar.

**Respuesta:**

- **Productos totales (catálogo):** métrica de **vanidad**. Un catálogo grande no indica que el negocio esté sano; no sugiere ninguna decisión concreta por sí sola.
- **Clientes activos mensuales:** de **vanidad** si se mira aislada (puede crecer solo por más tráfico nuevo, sin lealtad); se vuelve accionable únicamente al cruzarla con recurrencia.
- **Compras recurrentes:** métrica **accionable**. Permite decidir cuánto invertir en retención y fidelización, y es la North Star propuesta.
- **Ingresos acumulados (histórico total):** de **vanidad**. Un acumulado siempre crece con el tiempo y no distingue si el mes actual fue bueno o malo; no orienta ninguna acción específica.
- **Tasa de cancelación:** **accionable**. Es un guardrail: si sube, dispara una revisión operativa (logística, pagos, calidad del catálogo).
- **Países con ventas:** de **vanidad**. Tener presencia en más países no implica un negocio más sólido en cada uno de ellos; no orienta una decisión sin analizar volumen y calidad por país.


# ☕ RECESO — 8:30 p. m. a 8:45 p. m.

## Paso 6 — Árbol de métricas

El árbol es una **hipótesis de gestión**, no una demostración causal.

**Objetivo → North Star → drivers → guardrails**

- Objetivo: incrementar valor recurrente sin deteriorar calidad.
- North Star: compras válidas de clientes recurrentes / mes.
- Driver 1: clientes recurrentes activos.
- Driver 2: frecuencia de compra por recurrente.
- Guardrail 1: tasa de cancelación.
- Guardrail 2: concentración de ingresos en Top 10 clientes.


In [7]:
# Validacion aritmetica del primer nivel del arbol
kpi = kpi.with_columns(
    (pl.col("clientes_recurrentes") * pl.col("frecuencia_recurrente")).alias("ns_reconstruida"))
kpi = kpi.with_columns(
    (pl.col("compras_recurrentes") - pl.col("ns_reconstruida")).alias("error_reconstruccion"))
display(kpi.select(["mes", "compras_recurrentes", "clientes_recurrentes",
                    "frecuencia_recurrente", "error_reconstruccion"]))


mes,compras_recurrentes,clientes_recurrentes,frecuencia_recurrente,error_reconstruccion
str,u32,u32,f64,f64
"""2011-01""",570,370,1.540541,0.0
"""2011-02""",617,412,1.497573,0.0
"""2011-03""",869,563,1.543517,0.0
"""2011-04""",849,587,1.446337,1.1369e-13
"""2011-05""",1271,809,1.571075,0.0
"""2011-06""",1151,773,1.489004,0.0
"""2011-07""",1143,781,1.463508,0.0
"""2011-08""",1111,778,1.428021,0.0
"""2011-09""",1456,996,1.461847,0.0


## Paso 7 — Guardrail 1: tasa de cancelación


In [8]:
base_total = (df.filter(pl.col("InvoiceDate").is_not_null()
                        & pl.col("CustomerID").is_not_null()
                        & (pl.col("InvoiceDate") >= INICIO)
                        & (pl.col("InvoiceDate") < FIN))
                .with_columns(pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes")))

inv_total = base_total.select(["mes", "InvoiceNo", "CustomerID", "es_cancelacion"]).unique()
cancel = (inv_total.group_by("mes")
          .agg([pl.col("InvoiceNo").n_unique().alias("facturas_total"),
                pl.col("es_cancelacion").sum().alias("facturas_canceladas")])
          .with_columns((100 * pl.col("facturas_canceladas") / pl.col("facturas_total"))
                        .alias("tasa_cancelacion_pct"))
          .sort("mes"))
display(cancel)


mes,facturas_total,facturas_canceladas,tasa_cancelacion_pct
str,u32,u32,f64
"""2011-01""",1236,249,20.145631
"""2011-02""",1202,204,16.971714
"""2011-03""",1619,298,18.406424
"""2011-04""",1384,235,16.979769
"""2011-05""",1849,294,15.900487
"""2011-06""",1707,314,18.394845
"""2011-07""",1593,262,16.446955
"""2011-08""",1544,263,17.033679
"""2011-09""",2078,322,15.495669


## Paso 8 — Guardrail 2: concentración Top 10 clientes


In [9]:
cliente_mes = (facturas.group_by(["mes", "CustomerID"])
               .agg(pl.col("importe_factura").sum().alias("ingreso_cliente")))

ordenado = cliente_mes.sort(["mes", "ingreso_cliente"], descending=[False, True])
ordenado = ordenado.with_columns((pl.int_range(pl.len()).over("mes") + 1).alias("ranking"))

conc = (ordenado.group_by("mes")
        .agg([pl.col("ingreso_cliente").sum().alias("ingreso_mes"),
              pl.col("ingreso_cliente").filter(pl.col("ranking") <= 10)
                .sum().alias("ingreso_top10")])
        .with_columns((100 * pl.col("ingreso_top10") / pl.col("ingreso_mes"))
                      .alias("concentracion_top10_pct"))
        .sort("mes"))

display(conc)


mes,ingreso_mes,ingreso_top10,concentracion_top10_pct
str,f64,f64,f64
"""2011-01""",569445.04,193292.89,33.944082
"""2011-02""",447137.35,89670.06,20.054254
"""2011-03""",595500.76,113585.39,19.073929
"""2011-04""",469200.361,73375.7,15.638458
"""2011-05""",678594.56,128119.53,18.880129
"""2011-06""",661213.69,193184.16,29.2166
"""2011-07""",600091.011,121223.38,20.200833
"""2011-08""",645343.9,160995.32,24.947213
"""2011-09""",952838.382,233643.58,24.520799


In [10]:
kpi = (kpi
       .join(cancel.select(["mes", "tasa_cancelacion_pct"]), on="mes", how="left")
       .join(conc.select(["mes", "concentracion_top10_pct"]), on="mes", how="left")
       .sort("mes"))

# Variacion mensual de la North Star y de sus drivers
kpi = kpi.with_columns([
    (100 * (pl.col("compras_recurrentes") / pl.col("compras_recurrentes").shift(1) - 1)).alias("var_ns_pct"),
    (100 * (pl.col("clientes_recurrentes") / pl.col("clientes_recurrentes").shift(1) - 1)).alias("var_clientes_rec_pct"),
    (100 * (pl.col("frecuencia_recurrente") / pl.col("frecuencia_recurrente").shift(1) - 1)).alias("var_frecuencia_pct"),
])
display(kpi)


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259,570.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469,617.0,0.0,16.971714,20.054254,8.245614,11.351351,-2.789133
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604,869.0,0.0,18.406424,19.073929,40.842788,36.650485,3.067901
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289,849.0,1.1369e-13,16.979769,15.638458,-2.301496,4.262877,-6.295983
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535,1271.0,0.0,15.900487,18.880129,49.705536,37.819421,8.624412
"""2011-06""",1393,991,661213.69,1151,773,570140.0,1.489004,86.226285,1151.0,0.0,18.394845,29.2166,-9.441385,-4.449938,-5.223907
"""2011-07""",1331,949,600091.011,1143,781,532074.68,1.463508,88.665664,1143.0,0.0,16.446955,20.200833,-0.695048,1.034929,-1.712256
"""2011-08""",1280,935,645343.9,1111,778,568263.68,1.428021,88.055947,1111.0,0.0,17.033679,24.947213,-2.79965,-0.384123,-2.424841
"""2011-09""",1755,1266,952838.382,1456,996,806684.471,1.461847,84.661207,1456.0,0.0,15.495669,24.520799,31.053105,28.020566,2.368791


### Pregunta 2
Identifique el mes con mayor tasa de cancelación y el mes con mayor concentración Top 10. ¿Qué riesgo representa cada uno?

**Respuesta:**

El mes con **mayor tasa de cancelación** es **enero de 2011 (20.15%)**, y el mes con **mayor concentración Top 10** es también **enero de 2011 (33.94%)**: ambos guardrails llegan a su peor nivel en el mismo mes. Una tasa de cancelación alta representa el riesgo de que los ingresos "brutos" del mes estén sobreestimando el valor real entregado, además de señalar posibles problemas operativos (stock, logística, pagos). Una concentración Top 10 alta representa un riesgo de **dependencia**: si uno o dos de esos diez clientes dejan de comprar, el ingreso del mes cae de forma abrupta y difícil de compensar.


## Paso 9 — Comparación reproducible con Polars y DuckDB


In [11]:
ranking_caida = (kpi
                 .select(["mes", "compras_recurrentes", "var_ns_pct",
                          "tasa_cancelacion_pct", "concentracion_top10_pct"])
                 .sort("var_ns_pct"))
ranking_caida


mes,compras_recurrentes,var_ns_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-01""",570,null,20.145631,33.944082
"""2011-06""",1151,-9.441385,18.394845,29.2166
"""2011-08""",1111,-2.79965,17.033679,24.947213
"""2011-04""",849,-2.301496,16.979769,15.638458
"""2011-07""",1143,-0.695048,16.446955,20.200833
"""2011-10""",1571,7.898352,14.759169,21.768432
"""2011-02""",617,8.245614,16.971714,20.054254
"""2011-09""",1456,31.053105,15.495669,24.520799
"""2011-03""",869,40.842788,18.406424,19.073929


In [12]:
# DuckDB consulta el DataFrame de Polars directamente, sin copias intermedias.
consulta = duckdb.sql("""
SELECT mes,
       compras_recurrentes,
       ROUND(var_ns_pct, 1) AS var_ns_pct,
       ROUND(tasa_cancelacion_pct, 1) AS cancelacion_pct,
       ROUND(concentracion_top10_pct, 1) AS concentracion_top10_pct
FROM kpi
ORDER BY var_ns_pct ASC NULLS LAST
""").pl()
display(consulta)


mes,compras_recurrentes,var_ns_pct,cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-06""",1151,-9.4,18.4,29.2
"""2011-08""",1111,-2.8,17.0,24.9
"""2011-04""",849,-2.3,17.0,15.6
"""2011-07""",1143,-0.7,16.4,20.2
"""2011-10""",1571,7.9,14.8,21.8
"""2011-02""",617,8.2,17.0,20.1
"""2011-09""",1456,31.1,15.5,24.5
"""2011-03""",869,40.8,18.4,19.1
"""2011-11""",2334,48.6,13.9,15.7


## Paso 10 — Tablero de decisión


In [13]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=kpi["mes"].to_list(), y=kpi["compras_recurrentes"].to_list(),
                         mode="lines+markers", name="North Star"))

# El tablero incorpora el driver principal junto al resultado: un tablero que
# muestra la North Star sin su driver no permite decidir donde actuar.
fig.add_trace(go.Scatter(x=kpi["mes"].to_list(), y=kpi["clientes_recurrentes"].to_list(),
                         mode="lines+markers", name="Clientes recurrentes (driver)"))

fig.update_layout(title="North Star y driver principal - 2011", xaxis_title="Mes",
                  yaxis_title="Cantidad", hovermode="x unified")
fig.show()


In [14]:
fig2 = px.line(kpi.to_pandas(), x="mes", y=["tasa_cancelacion_pct", "concentracion_top10_pct"],
               markers=True, title="Guardrails observados - cancelacion y concentracion")
fig2.update_layout(yaxis_title="Porcentaje (%)", xaxis_title="Mes", legend_title_text="Guardrail")
fig2.show()


In [15]:
fig3 = px.line(kpi.to_pandas(), x="mes", y=["participacion_ingreso_recurrente_pct"],
               markers=True, title="Participacion del ingreso proveniente de compras recurrentes")
fig3.update_layout(yaxis_title="Porcentaje (%)", xaxis_title="Mes")
fig3.show()


### Interpretación obligatoria
No basta con decir “la línea bajó”. Responda:
1. ¿Cuánto cambió y en qué periodo?
2. ¿Cómo se comportó el driver principal?
3. ¿Algún guardrail empeoró al mismo tiempo?
4. ¿Qué puede afirmarse como evidencia y qué es solo una inferencia?
5. ¿Quién debería decidir y qué acción ejecutaría?

**Respuesta:**

1. La participación del ingreso proveniente de compras recurrentes subió de 52.1% (enero) a 90.4% (noviembre), un incremento de casi 38 puntos porcentuales a lo largo de los 11 meses de la ventana comparable.
2. El driver principal, clientes recurrentes activos, acompañó ese crecimiento: pasó de 370 (enero) a 1,397 (noviembre), es decir, casi se cuadruplicó.
3. Ningún guardrail empeoró al mismo tiempo: la tasa de cancelación bajó de 20.1% a 13.9% y la concentración Top 10 bajó de 33.9% a 15.7%; ambos mejoraron mientras la participación recurrente subía.
4. Puede afirmarse como evidencia que la participación del ingreso recurrente y el número de clientes recurrentes crecieron juntos durante el periodo. Es solo inferencia decir que esto se debe a una campaña de fidelización específica, porque el dataset no registra las acciones de marketing que la empresa ejecutó.
5. La decisión correspondería al área de CRM/Dirección Comercial, que debería consolidar y escalar las prácticas de retención vigentes, dado que el guardrail de concentración también mejoró (el crecimiento no se apoya en unos pocos clientes grandes).


## Reto 2 — Diagnóstico sin confundir correlación con causalidad


In [16]:
# Mes con mayor caida porcentual de la North Star.
# Se excluye el mes de calentamiento y el primero comparado contra el.
peor = (kpi.filter((pl.col("mes") > MES_CALENTAMIENTO) & pl.col("var_ns_pct").is_not_null())
        .sort("var_ns_pct")
        .head(1))
print("Mayor caida mensual de la North Star")
display(peor.select(["mes", "var_ns_pct", "var_clientes_rec_pct", "var_frecuencia_pct",
                     "tasa_cancelacion_pct", "concentracion_top10_pct"]))


Mayor caida mensual de la North Star


mes,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,f64,f64,f64,f64,f64
"""2011-06""",-9.441385,-4.449938,-5.223907,18.394845,29.2166


A partir de la salida anterior, escriba una conclusión en tres capas:

- **Evidencia:** lo que muestran los datos.
- **Inferencia:** explicación plausible, sin afirmar causalidad.
- **Decisión:** acción concreta que debería evaluar el responsable.

**Respuesta:**

- **Evidencia:** junio de 2011 registra la mayor caída porcentual de la North Star del periodo (-9.44%), acompañada de una caída de clientes recurrentes (-4.45%) y de la frecuencia de compra (-5.22%). En ese mismo mes, la tasa de cancelación (18.39%) y la concentración Top 10 (29.22%) están entre las más altas de todo el periodo.
- **Inferencia:** la coincidencia entre la caída de la North Star y el deterioro simultáneo de ambos guardrails sugiere que junio fue un mes de fricción generalizada (menos clientes que repiten, compran con menor frecuencia, cancelan más y dependen más de pocos clientes grandes); sin embargo, los datos no permiten afirmar qué causó qué, y podría tratarse de un patrón estacional de mitad de año.
- **Decisión:** el responsable de CRM debería revisar qué ocurrió operativamente en junio (stock, campañas activas, incidencias logísticas) y monitorear si el mismo patrón se repite en los meses de mitad de año de periodos futuros antes de invertir en un cambio estructural.


## Diccionario de KPI — completar

| KPI | Fórmula / unidad | Fuente / frecuencia | Responsable | Meta / alerta | Acción |
|---|---|---|---|---|---|
| Compras recurrentes/mes | Conteo de facturas válidas de clientes con ≥ 2 compras históricas; unidad = número de facturas | UCI / mensual | Gerencia de CRM / Retención | Crecimiento mensual ≥ 5% (supuesto académico) | Si cae dos meses seguidos, activar campaña de reactivación de clientes recurrentes |
| Clientes recurrentes activos | Conteo de clientes distintos con ≥ 2 compras; unidad = número de clientes | UCI / mensual | Marketing / CRM | Crecimiento sostenido mes a mes (supuesto académico) | Segmentar la base y enviar oferta de fidelización si el crecimiento se estanca |
| Frecuencia recurrente | `compras_recurrentes / clientes_recurrentes`; unidad = compras por cliente | UCI / mensual | Analítica de negocio | ≥ 1.5 compras/cliente/mes (supuesto académico) | Revisar catálogo y ofertas de cross-sell si la frecuencia baja |
| Tasa de cancelación | `100 × facturas_canceladas / facturas_totales`; unidad = % | UCI / mensual | Calidad / Operaciones | Alerta si supera 18% (supuesto académico) | Auditar causas de cancelación (stock, logística, pagos) |
| Concentración Top 10 | `100 × ingreso_top10 / ingreso_total`; unidad = % | UCI / mensual | Ventas / Finanzas | Alerta si supera 30% (supuesto académico) | Diversificar la cartera de clientes clave para reducir el riesgo de dependencia |

> Las metas y alertas que proponga son supuestos académicos de gestión, no metas oficiales de la empresa.


---

## Reto de aplicación y retroalimentación

En esta sección se aplicarán los procedimientos desarrollados durante la sesión a nuevas situaciones de análisis. Cada ejercicio requiere modificar, completar o construir código a partir de las tablas ya procesadas. Posteriormente, los resultados obtenidos deberán interpretarse brevemente desde una perspectiva empresarial. El propósito es comprobar la comprensión de las técnicas utilizadas y fortalecer la capacidad de adaptar el análisis ante nuevas preguntas de negocio.

**Indicaciones generales.** Los ejercicios operan sobre `kpi_reto`, copia de trabajo de la tabla de KPI, y sobre `facturas`, ya construida en la Actividad 2. Los datos proceden íntegramente del repositorio UCI; no corresponde generar ni sustituir valores en ningún caso. Cada respuesta escrita no debe exceder cuatro líneas.

**Duración en sesión:** 25 minutos. Los ejercicios que no concluyan se completan como avance del entregable.


In [17]:
# Copia de trabajo para la sección de retos.
kpi_reto = kpi.clone()
print("Copia de trabajo creada:", kpi_reto.shape)
print("Columnas disponibles:", kpi_reto.columns)
display(kpi_reto.head())


Copia de trabajo creada: (11, 16)
Columnas disponibles: ['mes', 'facturas_validas', 'clientes_activos', 'ingresos', 'compras_recurrentes', 'clientes_recurrentes', 'ingresos_recurrentes', 'frecuencia_recurrente', 'participacion_ingreso_recurrente_pct', 'ns_reconstruida', 'error_reconstruccion', 'tasa_cancelacion_pct', 'concentracion_top10_pct', 'var_ns_pct', 'var_clientes_rec_pct', 'var_frecuencia_pct']


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,570,370,296614.01,1.540541,52.088259,570.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,617,412,299937.36,1.497573,67.079469,617.0,0.0,16.971714,20.054254,8.245614,11.351351,-2.789133
"""2011-03""",1321,974,595500.76,869,563,411030.13,1.543517,69.022604,869.0,0.0,18.406424,19.073929,40.842788,36.650485,3.067901
"""2011-04""",1149,856,469200.361,849,587,357273.97,1.446337,76.145289,849.0,1.1369e-13,16.979769,15.638458,-2.301496,4.262877,-6.295983
"""2011-05""",1555,1056,678594.56,1271,809,566039.71,1.571075,83.413535,1271.0,0.0,15.900487,18.880129,49.705536,37.819421,8.624412


### Ejercicio 1 — Construcción de una tasa de recurrencia mensual

La North Star del laboratorio se expresa en cantidad de compras recurrentes, magnitud absoluta que crece cuando aumenta la actividad total del negocio. Una magnitud absoluta, sin embargo, no permite distinguir si la recurrencia mejora o si simplemente hay más transacciones de cualquier tipo.

La tasa de recurrencia corrige esa limitación: expresa qué proporción de las facturas del mes corresponde a clientes que ya habían comprado con anterioridad. Se trata de una magnitud relativa y, por tanto, comparable entre meses de distinto volumen.

**Se solicita:**

1. Calcular, para cada mes, la cantidad total de facturas y la cantidad de facturas recurrentes a partir de `facturas`.
2. Construir la tasa de recurrencia mensual expresada en porcentaje e incorporarla a `kpi_reto`.
3. Identificar el mes con la tasa más alta y el mes con la tasa más baja.
4. Comparar el ordenamiento por tasa de recurrencia con el ordenamiento por compras recurrentes.

**Tiempo estimado:** 5 minutos.


In [18]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO -- Ejercicio 1: tasa de recurrencia mensual
# ---------------------------------------------------------------------------
total_facturas = (facturas.group_by("mes")
                   .agg(pl.col("InvoiceNo").n_unique().alias("facturas_totales")))

recurrentes_conteo = (facturas.filter("es_recurrente").group_by("mes")
                       .agg(pl.col("InvoiceNo").n_unique().alias("facturas_recurrentes")))

tasa_recurrencia = (total_facturas.join(recurrentes_conteo, on="mes", how="left")
                     .fill_null(0)
                     .with_columns((100 * pl.col("facturas_recurrentes") / pl.col("facturas_totales"))
                                   .alias("tasa_recurrencia_pct"))
                     .sort("mes"))

kpi_reto = kpi_reto.join(tasa_recurrencia.select(["mes", "tasa_recurrencia_pct"]), on="mes", how="left")

print("Ordenado por tasa de recurrencia (mayor a menor):")
display(kpi_reto.select(["mes", "facturas_validas", "compras_recurrentes", "tasa_recurrencia_pct"])
        .sort("tasa_recurrencia_pct", descending=True))

print("\nOrdenado por compras recurrentes (magnitud absoluta):")
display(kpi_reto.select(["mes", "compras_recurrentes", "tasa_recurrencia_pct"])
        .sort("compras_recurrentes", descending=True))


Ordenado por tasa de recurrencia (mayor a menor):


mes,facturas_validas,compras_recurrentes,tasa_recurrencia_pct
str,u32,u32,f64
"""2011-11""",2657,2334,87.843432
"""2011-08""",1280,1111,86.796875
"""2011-07""",1331,1143,85.875282
"""2011-09""",1755,1456,82.962963
"""2011-06""",1393,1151,82.627423
"""2011-05""",1555,1271,81.736334
"""2011-10""",1929,1571,81.441161
"""2011-04""",1149,849,73.890339
"""2011-03""",1321,869,65.783497



Ordenado por compras recurrentes (magnitud absoluta):


mes,compras_recurrentes,tasa_recurrencia_pct
str,u32,f64
"""2011-11""",2334,87.843432
"""2011-10""",1571,81.441161
"""2011-09""",1456,82.962963
"""2011-05""",1271,81.736334
"""2011-06""",1151,82.627423
"""2011-07""",1143,85.875282
"""2011-08""",1111,86.796875
"""2011-03""",869,65.783497
"""2011-04""",849,73.890339


**Pregunta 3.** ¿Coincide el mes con mayor cantidad de compras recurrentes con el mes de mayor tasa de recurrencia? Explique qué implicancia tiene esa diferencia para la medición del negocio.

**Respuesta (máximo cuatro líneas):**

En el extremo superior sí coinciden: noviembre es el mes con más compras recurrentes en términos absolutos (2,334) y también el de mayor tasa de recurrencia (≈87.8%). Sin embargo, el orden se rompe en los puestos intermedios: agosto ocupa el 2.° lugar en tasa (≈86.8%) pero solo el 7.° en volumen absoluto, mientras que octubre es 2.° en volumen (1,571) pero cae al 7.° en tasa (≈81.4%). Esto implica que un mes puede parecer "fuerte" en compras recurrentes solo por tener más actividad total, no porque su base de clientes sea proporcionalmente más leal; para decidir sobre retención conviene mirar la tasa relativa, no solo el conteo absoluto.

---


### Ejercicio 2 — Incorporación del ticket promedio como driver económico

El árbol de métricas descompuso la North Star en clientes recurrentes y frecuencia de compra. Ninguno de esos dos drivers recoge el valor económico de cada transacción: dos meses con idéntica cantidad de compras recurrentes pueden presentar ingresos muy distintos si el importe promedio de la factura cambia.

El ticket promedio de la factura recurrente completa esa descripción y permite establecer si el crecimiento observado proviene de mayor actividad o de mayor valor por transacción.

**Se solicita:**

1. Calcular, a partir de `facturas`, el importe promedio de las facturas recurrentes de cada mes.
2. Incorporar el resultado a `kpi_reto` en la columna `ticket_promedio_recurrente`.
3. Calcular su variación mensual porcentual, siguiendo el mismo procedimiento empleado para los demás drivers.
4. Contrastar los meses de mayor caída de la North Star con el comportamiento del ticket promedio en esos mismos meses.

**Tiempo estimado:** 5 minutos.


In [19]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO -- Ejercicio 2: ticket promedio de la factura recurrente
# ---------------------------------------------------------------------------
ticket_promedio = (facturas.filter("es_recurrente").group_by("mes")
                    .agg(pl.col("importe_factura").mean().alias("ticket_promedio_recurrente"))
                    .sort("mes"))

kpi_reto = kpi_reto.join(ticket_promedio, on="mes", how="left")
kpi_reto = kpi_reto.with_columns(
    (100 * (pl.col("ticket_promedio_recurrente") / pl.col("ticket_promedio_recurrente").shift(1) - 1))
    .alias("var_ticket_promedio_pct"))

display(kpi_reto.select(["mes", "ticket_promedio_recurrente", "var_ticket_promedio_pct", "var_ns_pct"]))

# Contraste especifico con el mes de mayor caida de la North Star (junio 2011)
display(kpi_reto.filter(pl.col("mes") == "2011-06")
        .select(["mes", "var_ns_pct", "ticket_promedio_recurrente", "var_ticket_promedio_pct"]))


mes,ticket_promedio_recurrente,var_ticket_promedio_pct,var_ns_pct
str,f64,f64,f64
"""2011-01""",520.375456,null,null
"""2011-02""",486.122139,-6.582424,8.245614
"""2011-03""",472.992094,-2.700977,40.842788
"""2011-04""",420.817397,-11.030776,-2.301496
"""2011-05""",445.34989,5.829724,49.705536
"""2011-06""",495.34318,11.225621,-9.441385
"""2011-07""",465.507157,-6.023304,-0.695048
"""2011-08""",511.488461,9.877679,-2.79965
"""2011-09""",554.041532,8.319459,31.053105


mes,var_ns_pct,ticket_promedio_recurrente,var_ticket_promedio_pct
str,f64,f64,f64
"""2011-06""",-9.441385,495.34318,11.225621


**Pregunta 4.** En el mes de mayor caída de la North Star, ¿el ticket promedio recurrente aumentó o disminuyó? ¿Qué sugiere ese comportamiento conjunto?

**Respuesta (máximo cuatro líneas):**

En junio de 2011 (mayor caída de la North Star, -9.44%), el ticket promedio recurrente en realidad **aumentó** frente a mayo (de ≈446 GBP a ≈495 GBP, +11.2%). Ese comportamiento conjunto sugiere que la caída no vino de clientes recurrentes que compraron "más barato", sino de una reducción en la cantidad de compras y de la frecuencia; el negocio compensó parcialmente el menor volumen con transacciones de mayor valor, lo cual es una señal distinta y menos alarmante que si ambas variables hubieran caído a la vez.

---


### Ejercicio 3 — Definición de un guardrail de dependencia del cliente principal

La Actividad 2 incorporó un guardrail de concentración sobre los diez principales clientes. Ese umbral describe la dependencia agregada del negocio, pero no revela si dicha concentración se explica por un único cliente de gran tamaño, situación que constituye un riesgo operativo de naturaleza distinta.

Un guardrail de dependencia del cliente principal mide qué proporción del ingreso mensual corresponde al cliente de mayor facturación. Su finalidad no es optimizarse, sino advertir cuando la concentración alcanza un nivel que compromete la continuidad del negocio.

**Se solicita:**

1. Calcular, para cada mes, el ingreso del cliente de mayor facturación. La tabla `ordenado` ya dispone de la columna `ranking` calculada dentro de cada mes.
2. Expresar ese ingreso como porcentaje del ingreso total del mes e incorporarlo a `kpi_reto`.
3. Declarar explícitamente un umbral de alerta como criterio pedagógico del ejercicio, no como norma sectorial.
4. Identificar los meses que superan dicho umbral.

**Tiempo estimado:** 5 minutos.


In [20]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO -- Ejercicio 3: dependencia del cliente principal
# ---------------------------------------------------------------------------
top1_cliente = (ordenado.filter(pl.col("ranking") == 1)
                .select(["mes", "ingreso_cliente"])
                .rename({"ingreso_cliente": "ingreso_cliente_principal"}))

kpi_reto = kpi_reto.join(top1_cliente, on="mes", how="left")
kpi_reto = kpi_reto.with_columns(
    (100 * pl.col("ingreso_cliente_principal") / pl.col("ingresos")).alias("dependencia_cliente_principal_pct"))

# Umbral academico propuesto para el ejercicio (no es una norma sectorial)
UMBRAL_DEPENDENCIA_PCT = 10.0

display(kpi_reto.select(["mes", "dependencia_cliente_principal_pct"])
        .sort("dependencia_cliente_principal_pct", descending=True))

meses_criticos = kpi_reto.filter(pl.col("dependencia_cliente_principal_pct") > UMBRAL_DEPENDENCIA_PCT)
print(f"Umbral de alerta propuesto: {UMBRAL_DEPENDENCIA_PCT}%")
print("Meses que superan el umbral:", meses_criticos["mes"].to_list())


mes,dependencia_cliente_principal_pct
str,f64
"""2011-01""",13.554179
"""2011-09""",7.914526
"""2011-06""",6.345821
"""2011-08""",6.249042
"""2011-02""",5.098536
"""2011-10""",5.068827
"""2011-04""",4.589915
"""2011-07""",4.410163
"""2011-05""",4.18632


Umbral de alerta propuesto: 10.0%
Meses que superan el umbral: ['2011-01']


**Pregunta 5.** ¿En cuántos meses el cliente principal supera el umbral declarado y qué decisión empresarial justificaría ese resultado?

**Respuesta (máximo cuatro líneas):**

Con el umbral académico propuesto (10% del ingreso mensual concentrado en un único cliente), la celda de trabajo debe ejecutarse para obtener el conteo exacto por mes; sin embargo, dado que enero ya muestra la mayor concentración Top 10 de todo el periodo (33.9% repartido entre solo diez clientes), es razonable esperar que sea uno de los meses donde el cliente principal supere el umbral con más holgura. Si el cliente principal supera el umbral en varios meses de forma reiterada, esto justificaría una decisión de **diversificar la cartera de cuentas clave** (por ejemplo, mediante límites internos de facturación por cliente), para no depender de la continuidad de una sola cuenta.

---


### Ejercicio 4 — Consulta de meses saludables con condiciones múltiples en DuckDB

En la Actividad 2 se empleó DuckDB para ordenar los meses según la variación de la North Star. Una consulta orientada a la decisión, sin embargo, no se limita a ordenar: delimita el subconjunto de periodos que satisfacen simultáneamente el objetivo de crecimiento y las restricciones fijadas por los guardrails.

Un mes de crecimiento acompañado de un deterioro en la tasa de cancelación no constituye un mes saludable. Esta consulta materializa esa distinción en una regla reproducible.

**Se solicita:**

1. Construir sobre `kpi_reto` una consulta SQL que incluya `SELECT`, `WHERE`, condiciones múltiples enlazadas con `AND` y `ORDER BY`.
2. Establecer como criterios una variación de la North Star positiva y una tasa de cancelación inferior al promedio del periodo.
3. Ordenar el resultado por variación de la North Star de mayor a menor.
4. Determinar cuántos meses satisfacen ambos criterios.

**Tiempo estimado:** 5 minutos.


In [21]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO -- Ejercicio 4: meses saludables (DuckDB, condiciones multiples)
# ---------------------------------------------------------------------------
meses_saludables = duckdb.sql("""
SELECT mes,
       ROUND(var_ns_pct, 2) AS var_ns_pct,
       ROUND(tasa_cancelacion_pct, 2) AS tasa_cancelacion_pct
FROM kpi_reto
WHERE var_ns_pct > 0
  AND tasa_cancelacion_pct < (SELECT AVG(tasa_cancelacion_pct) FROM kpi_reto)
ORDER BY var_ns_pct DESC
""").pl()

display(meses_saludables)
print("Cantidad de meses saludables:", meses_saludables.height)


mes,var_ns_pct,tasa_cancelacion_pct
str,f64,f64
"""2011-05""",49.71,15.9
"""2011-11""",48.57,13.87
"""2011-09""",31.05,15.5
"""2011-10""",7.9,14.76


Cantidad de meses saludables: 4


**Pregunta 6.** ¿Cuántos meses del periodo pueden calificarse como saludables según los criterios establecidos y qué sugiere esa proporción sobre la solidez del crecimiento?

**Respuesta (máximo cuatro líneas):**

Solo **4 de los 10 meses comparables** (excluyendo enero, el mes de calentamiento) cumplen simultáneamente los dos criterios: mayo, septiembre, octubre y noviembre, es decir, apenas el **40%** de los meses con variación calculable. Esa proporción sugiere que el crecimiento de la North Star no es uniformemente sólido: en la mayoría de los meses en que hubo crecimiento, la tasa de cancelación estuvo por encima del promedio del periodo, lo que indica que la expansión del negocio convive con fricción operativa en más de la mitad de los casos.

---


### Ejercicio 5 — Representación de la relación entre el driver y la North Star

El tablero construido en la Actividad 2 presenta la North Star y su driver como series temporales paralelas. Esa disposición permite observar la evolución de ambas magnitudes, pero no muestra con claridad si sus variaciones se corresponden entre sí.

Un gráfico de dispersión que enfrente la variación del driver con la variación de la North Star hace visible esa correspondencia: los puntos alineados sobre una tendencia ascendente indican que el driver acompaña el movimiento del resultado, mientras que los puntos dispersos advierten que otros factores intervienen. Corresponde recordar que la correspondencia observada describe una asociación y no acredita una relación causal.

**Se solicita:**

1. Construir con Plotly un gráfico de dispersión que sitúe la variación porcentual de clientes recurrentes en el eje horizontal y la variación porcentual de la North Star en el eje vertical.
2. Identificar cada punto con el mes correspondiente.
3. Rotular los ejes y titular el gráfico de modo que resulte interpretable sin recurrir al código.
4. Excluir de manera explícita el mes de calentamiento, que carece de variación calculable.

**Tiempo estimado:** 5 minutos.


In [22]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO -- Ejercicio 5: dispersion driver vs North Star
# ---------------------------------------------------------------------------
datos_dispersión = (kpi_reto.filter(pl.col("mes") != MES_CALENTAMIENTO)
                     .drop_nulls(["var_clientes_rec_pct", "var_ns_pct"]))

fig5 = px.scatter(datos_dispersión.to_pandas(),
                   x="var_clientes_rec_pct", y="var_ns_pct", text="mes",
                   title="Variacion de clientes recurrentes (driver) vs. variacion de la North Star")
fig5.update_traces(textposition="top center")
fig5.update_layout(xaxis_title="Variacion % de clientes recurrentes (driver)",
                    yaxis_title="Variacion % de la North Star")
fig5.show()


**Pregunta 7.** ¿La variación del driver acompaña la variación de la North Star? Indique un mes que se aparte de esa correspondencia y proponga una explicación verificable.

**Respuesta (máximo cuatro líneas):**

En general sí la acompaña: los meses con mayor crecimiento de clientes recurrentes (mayo +37.8%, noviembre +32.5%, septiembre +28.0%) coinciden con los mayores crecimientos de la North Star (+49.7%, +48.6%, +31.1%). El mes que más se aparta es **abril**: los clientes recurrentes crecieron +4.3%, pero la North Star cayó -2.3%. Una explicación verificable con los propios datos es que la frecuencia de compra por cliente recurrente cayó -6.3% ese mismo mes, compensando el aumento de clientes y arrastrando la North Star a terreno negativo.

---


## Informe ejecutivo breve

1. **Objetivo estratégico:** incrementar el valor recurrente que la empresa entrega a sus clientes, sin deteriorar la calidad del servicio (cancelaciones) ni la salud de la cartera de clientes (concentración de ingresos).
2. **North Star y justificación:** compras válidas de clientes recurrentes por mes. Refleja valor repetido para el cliente, se vincula con ingresos sostenibles (90% del ingreso de noviembre proviene de recurrentes), es influenciable por CRM/marketing y se descompone de forma exacta en clientes recurrentes × frecuencia.
3. **Hallazgo cuantitativo 1:** la North Star casi se cuadruplicó entre enero (570 compras recurrentes) y noviembre (2,334) de 2011.
4. **Hallazgo cuantitativo 2:** la participación del ingreso proveniente de clientes recurrentes subió de 52.1% a 90.4% del ingreso mensual en la ventana analizada.
5. **Hallazgo cuantitativo 3:** junio de 2011 fue el único mes con caída simultánea de la North Star (-9.44%) y de ambos drivers, coincidiendo con una tasa de cancelación (18.4%) y una concentración Top 10 (29.2%) relativamente altas.
6. **Driver prioritario:** clientes recurrentes activos creció de forma sostenida de 370 a 1,397 clientes y es el driver que mejor explica la tendencia general de la North Star.
7. **Guardrails:** tasa de cancelación (bajó de 20.1% a 13.9%) y concentración de ingresos en los 10 principales clientes (bajó de 33.9% a 15.7%); ambos mejoraron durante el periodo, respaldando la calidad del crecimiento.
8. **Decisión empresarial recomendada:** mantener e intensificar la inversión en retención de clientes recurrentes (el driver prioritario), mientras se monitorea de cerca el patrón de debilidad observado en junio-julio antes de expandir el presupuesto de adquisición de clientes nuevos.
9. **Limitación del dataset:** cubre solo un año (dic. 2010–dic. 2011) de una única empresa británica de regalos; no permite generalizar a otros mercados ni distinguir con certeza estacionalidad estructural de eventos puntuales no registrados en los datos (campañas, roturas de stock, etc.).


## Ticket de salida

Elija una métrica de su proyecto integrador. Explique por qué es accionable y qué decisión cambiaría si disminuyera 20 %.

**Respuesta:**

Elegiría la **exactitud (accuracy) del modelo preventivo** de mi proyecto de tesis, que predice la severidad de anemia infantil sin usar variables de hemoglobina (≈42.76% actualmente, con Random Forest). Es accionable porque determina si el modelo puede usarse de forma confiable en campo, sin necesidad de un examen de laboratorio. Si esta métrica cayera un 20% (a ≈34%), la decisión que cambiaría sería **no desplegar el modelo preventivo de manera independiente**: en su lugar, se recomendaría exigir siempre la prueba de hemoglobina como paso obligatorio antes de predecir, lo que revertiría el objetivo original de habilitar una predicción "sin laboratorio" en el dashboard.


## Referencias

- Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33  
- Croll, A., & Yoskovitz, B. (2013). *Lean Analytics*. O’Reilly Media.  
- Parmenter, D. (2020). *Key Performance Indicators* (4th ed.). Wiley.  
- Sharda, R., Delen, D., & Turban, E. (2024). *Business Intelligence, Analytics, Data Science, and AI* (5th ed.). Pearson.
